# tttk: perillas del TTT medidas con el instrumento de deficit (30-oct)

Pares exactos por tarea dentro del mismo worker. Variantes del calendario del TTT (lr, epocas, nº de permutaciones, semilla). Tras cada TTT: teacher forcing de la GT en la vista canonica + 3 vistas aumentadas fijas: deficit vs greedy, ancho implicado, NLL. `base_s2` calibra el ruido del propio instrumento. Sin decode. 4 GPUs (1 worker por GPU).


In [ ]:
%%writefile /kaggle/working/_worker_src.py
import glob, hashlib, json, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()

SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "4"))
N_TAREAS = int(os.environ.get("N_TAREAS", "12"))
N_VISTAS = int(os.environ.get("N_VISTAS", "16"))
BUDGET = float(os.environ.get("BUDGET_VISTA", "45"))
WALL_MAX = float(os.environ.get("WALL_MAX", "2100"))
# UN SOLO BRAZO = produccion exacta. Las tareas las asigna el lanzador (plan LPT por coste).
BRAZO = "r256"
R_LORA = 256
ALPHA = 32
SUCIAS = set(json.loads(os.environ["SUCIAS"]))
OUT = "/kaggle/working/shard%d.json" % SHARD

import numpy as np
import torch
from peft import (LoraConfig, get_peft_model, get_peft_model_state_dict,
                  set_peft_model_state_dict)
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

# --- gate de carga: aborta ESTE shard, no la maquina ---
FICH = "arc-agi_evaluation_challenges.json"
DATA = None
for cand in ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
             "/kaggle/input/arc-prize-2026-arc-agi-2"):
    if os.path.exists(os.path.join(cand, FICH)):
        DATA = cand
        break
if DATA is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if FICH in files:
            DATA = root
            break
assert DATA, "sin dataset de la competicion"
CHAL = json.load(open(os.path.join(DATA, FICH)))
SOLS = json.load(open(os.path.join(DATA, "arc-agi_evaluation_solutions.json")))

MDIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(x.endswith(".safetensors") for x in files):
        MDIR = root
        break
assert MDIR, "no se encontro el modelo en /kaggle/input"
print("[s%d] modelo en %s" % (SHARD, MDIR), flush=True)

_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))
MAX_SEQ = 3500
LR = 5e-5


def coste(t):
    return sum(len(g["input"]) * len(g["input"][0]) + len(g["output"]) * len(g["output"][0])
               for g in t["train"])


# poblacion ESTRATIFICADA por coste a lo largo de TODA la distribucion: coger las N mas
# baratas fue el modo de fallo de lb33-cap (-2,5 pt). Indices equiespaciados en el ranking.
MIS_KEYS = json.loads(os.environ["MIS_KEYS"])
VACIAS = set(json.loads(os.environ.get("VACIAS", "[]")))
VARIANTES = json.loads(os.environ["VARIANTES"])
N_VIS_MED = int(os.environ.get("N_VIS_MED", "3"))
print("[s%d] produccion (r=%d alpha=%d) | %d tareas | costes %s"
      % (SHARD, R_LORA, ALPHA, len(MIS_KEYS), [coste(CHAL[k]) for k in MIS_KEYS]), flush=True)


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def d4_inv(g, k):
    a = np.array(g)
    for _ in range(k % 4):
        a = np.rot90(a, -1)
    if k >= 4:
        a = a.T
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def invertir(g, k, perm):
    inv = [0] * 10
    for i, p in enumerate(perm):
        inv[p] = i
    return d4_inv([[inv[c] for c in fila] for fila in g], k)


def build_vistas(pares, seed, n):
    # vistas de DECODE: produccion usa augment(n=2) = 8 D4 x 2 perms = 16 vistas.
    rng = random.Random(seed)
    vistas = []
    for v in range(n):
        if v == 0:
            k, perm, orden = 0, list(range(10)), list(range(len(pares)))
        else:
            k = rng.randrange(8)
            perm = rng.sample(range(10), 10)
            orden = rng.sample(range(len(pares)), len(pares))
        vistas.append({"k": k, "perm": perm, "orden": orden})
    return vistas


def build_vistas_train(pares, seed, n_perm=16):
    # vistas de TTT: produccion usa augment(n=16) = 8 transformaciones D4 x 16
    # permutaciones de color = 128 SECUENCIAS, cada una vista UNA vez (1 epoca).
    # El PRODUCTO CRUZADO importa: 128 pasos, no 16. La primera version de este banco
    # generaba 16 vistas sueltas -> 8x menos adaptacion -> el 4B no aprendia nada
    # (3 de 16 tareas con GT en alguna vista). El smoke del 2B no lo detecto porque
    # ese modelo memoriza el eval y resuelve sin necesitar TTT.
    rng = random.Random(seed)
    perms = [list(range(10))] + [rng.sample(range(10), 10) for _ in range(n_perm - 1)]
    vistas = []
    for k in range(8):
        for perm in perms:
            vistas.append({"k": k, "perm": perm,
                           "orden": rng.sample(range(len(pares)), len(pares))})
    return vistas


base = AutoModelForCausalLM.from_pretrained(
    MDIR, torch_dtype=torch.bfloat16, device_map="cuda:0", attn_implementation="sdpa")
LCFG = LoraConfig(r=R_LORA, lora_alpha=ALPHA, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
INIT = {k: v.detach().clone() for k, v in get_peft_model_state_dict(MODEL).items()}
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:
    OPT_CLS = torch.optim.AdamW
print("[s%d] modelo listo mem=%.1fGB (%.0fs)"
      % (SHARD, torch.cuda.memory_allocated() / 1e9, time.time() - T00), flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs(prompt, depth, budget):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves


def _prompt_vista(task, v):
    p = []
    for idx in v["orden"]:
        par = task["train"][idx]
        t, _ = pair_tokens_mask(aplicar(par["input"], v["k"], v["perm"]), aplicar(par["output"], v["k"], v["perm"]))
        p += t
    return p + query_tokens(aplicar(task["test"][0]["input"], v["k"], v["perm"]))


@torch.no_grad()
def rangos_de_gt(task, gt, v):
    """EL INSTRUMENTO CENTRAL, y cuesta UN forward: fuerza la GT y mide, en cada posicion,
    el RANGO del token correcto entre los 12 tokens ARC y su NLL. De ahi sale analiticamente
    que ancho de haz haria falta:
      - rango 1 en todas las posiciones  => el greedy YA produce la GT (ancho 1 basta)
      - deficit acumulado frente al camino greedy => cota del ancho necesario (exp(deficit))"""
    prompt = _prompt_vista(task, v)
    gtoks = grid_tokens(aplicar(gt, v["k"], v["perm"])) + [EOS]
    ids = torch.tensor([prompt + gtoks], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, use_cache=False, return_dict=True)
    lg = out.logits[0, len(prompt) - 1:len(prompt) - 1 + len(gtoks)].float()
    lg = lg.index_select(-1, _TOKID)                      # solo los 12 tokens ARC
    lp = lg.log_softmax(-1)
    idx = {t: i for i, t in enumerate(ARC_TOKENS)}
    col = torch.tensor([idx[t] for t in gtoks], device="cuda:0")
    nll_gt = -lp.gather(1, col.view(-1, 1)).squeeze(1)     # NLL del token correcto
    nll_best = -lp.max(dim=1).values                       # NLL del argmax
    rango = (lp > lp.gather(1, col.view(-1, 1))).sum(1) + 1
    deficit = float((nll_gt - nll_best).sum().item())      # cuanto peor es la GT que el greedy
    r = rango.tolist()
    return {"n_tok": len(gtoks), "n_no_argmax": int(sum(1 for x in r if x > 1)),
            "rango_max": int(max(r)), "rangos_altos": sorted([x for x in r if x > 1], reverse=True)[:8],
            "deficit_vs_greedy": round(deficit, 3),
            "ancho_implicado": round(float(np.exp(min(deficit, 20))), 1),
            "nll_total": round(float(nll_gt.sum().item()), 3)}


def _reordena(cache, idx):
    """Reordena/duplica las filas de la cache KV para que coincidan con los padres elegidos.
    Sin esto el haz mezcla la historia de unas secuencias con los tokens de otras (fallo
    silencioso). Se prueban las tres APIs que ha tenido transformers; si ninguna existe se
    LANZA, no se ignora."""
    if hasattr(cache, "batch_select_indices"):
        cache.batch_select_indices(idx)
        return
    if getattr(cache, "key_cache", None) is not None:
        for i in range(len(cache.key_cache)):
            cache.key_cache[i] = cache.key_cache[i].index_select(0, idx)
            cache.value_cache[i] = cache.value_cache[i].index_select(0, idx)
        return
    capas = getattr(cache, "layers", None)
    if capas:
        for c in capas:
            c.keys = c.keys.index_select(0, idx)
            c.values = c.values.index_select(0, idx)
        return
    raise RuntimeError("no se como reordenar esta cache: %s" % type(cache))


@torch.no_grad()
def haz(task, gt, ancho, tope_tok):
    """Haz de ancho fijo, VECTORIZADO. La version anterior hacia 96 llamadas a .item() por
    paso (96 sincronizaciones GPU->CPU, hasta 89.000 por tarea larga) y costaba 181 s en la
    tarea mas larga contra un suelo teorico de ~19 s. Aqui: una ordenacion en GPU y DOS
    .tolist() por paso. Ademas el prefill va con lote 1 y la cache se expande, en vez de
    recomputar el prompt `ancho` veces."""
    prompt = _prompt_canonico(task)
    gtoks = grid_tokens(gt) + [EOS]
    NT = len(ARC_TOKENS)
    IEOS = ARC_TOKENS.index(EOS)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=torch.tensor([prompt], device="cuda:0", dtype=torch.long),
                    use_cache=True, return_dict=True)
    cache = out.past_key_values
    lp = out.logits[:, -1].float().index_select(-1, _TOKID).log_softmax(-1)     # [1, NT]
    sc = torch.zeros(1, device="cuda:0")
    caminos = [[]]
    acabados = []
    for _ in range(tope_tok):
        tot = (sc.view(-1, 1) - lp).reshape(-1)               # [k*NT] NLL acumulada
        orden = torch.argsort(tot)
        vals = tot.index_select(0, orden).tolist()            # 1 sincronizacion
        idxs = orden.tolist()                                 # 1 sincronizacion
        nuevos, padres, toks, nsc = [], [], [], []
        for v, ix in zip(vals, idxs):
            if len(nuevos) >= ancho:
                break
            p, j = ix // NT, ix % NT
            cam = caminos[p] + [ARC_TOKENS[j]]
            if j == IEOS:
                acabados.append((v, cam))
                continue
            if len(cam) >= tope_tok:
                continue
            nuevos.append(cam); padres.append(p); toks.append(ARC_TOKENS[j]); nsc.append(v)
        if not nuevos or len(acabados) >= ancho:
            break
        _reordena(cache, torch.tensor(padres, device="cuda:0", dtype=torch.long))
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor(toks, device="cuda:0", dtype=torch.long).view(-1, 1),
                      past_key_values=cache, use_cache=True, return_dict=True)
        cache = o.past_key_values
        lp = o.logits[:, -1].float().index_select(-1, _TOKID).log_softmax(-1)
        sc = torch.tensor(nsc, device="cuda:0")
        caminos = nuevos
    acabados.sort(key=lambda x: x[0])
    rejillas = []
    for v, c in acabados[:ancho]:
        g = toks_to_grid(c[:-1] if c and c[-1] == EOS else c)
        if g is not None:
            rejillas.append({"grid": g, "score": round(v, 4)})
    return {"ancho": ancho, "n_acabados": len(acabados), "gt_en_haz": any(c == gtoks for _, c in acabados),
            "cands": rejillas[:8],                      # se guardan para re-jugar la seleccion offline
            "mejor_ok": bool(rejillas and rejillas[0]["grid"] == gt)}


@torch.no_grad()
def nll_de_gt(task, gt):
    """NLL de la rejilla correcta bajo el modelo YA ADAPTADO, vista canonica (k=0, perm
    identidad). Medida continua: mas potencia que el 0-16 con n=12. Es MECANISMO, no
    veredicto -- no confundir con la loss del TTT, que esta saturada."""
    prompt = []
    for p in task["train"]:
        t, _ = pair_tokens_mask(p["input"], p["output"])
        prompt += t
    prompt += query_tokens(task["test"][0]["input"])
    gtoks = grid_tokens(gt) + [EOS]
    ids = torch.tensor([prompt + gtoks], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, use_cache=False, return_dict=True)
    lg = out.logits[0, :-1].float()
    tgt = ids[0, 1:]
    lp = lg.log_softmax(-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    n_p = len(prompt)
    return float(-(lp[n_p - 1:]).sum().item()), len(gtoks)


def entrenar_ttt(task, vistas_train, lr=LR, epocas=1, mask_all=False, seed=0):
    # UNA epoca sobre las vistas: calendario EXACTO de produccion (128 secuencias vistas
    # una vez). Recortarlo es el modo de fallo de ttt05 (26,39) -- no se toca.
    MODEL.config.use_cache = False
    MODEL.train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=lr)
    seqs = []
    for v in vistas_train:
        toks, mask = [], []
        for idx in v["orden"]:
            p = task["train"][idx]
            t, m = pair_tokens_mask(aplicar(p["input"], v["k"], v["perm"]),
                                    aplicar(p["output"], v["k"], v["perm"]))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            if mask_all:
                mask = [0 if t_ in (IM_START, USER, ASSIST) else 1 for t_ in toks]
            seqs.append((toks, mask))
    orden = list(seqs)
    rng_ep = random.Random(seed + 7919)
    for ep in range(1, epocas):
        extra = list(seqs); rng_ep.shuffle(extra); orden += extra
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(len(orden), 1))
    for toks, mask in orden:
        ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
        lab = torch.tensor([[t if m else -100 for t, m in zip(toks, mask)]],
                           device="cuda:0", dtype=torch.long)
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=ids, labels=lab)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in MODEL.parameters() if p.requires_grad], 1.0)
        opt.step()
        sch.step()
        opt.zero_grad(set_to_none=True)
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    MODEL.eval()
    MODEL.config.use_cache = True
    return len(seqs)


RES = {"tareas": [], "modelo": "4b", "n_vistas": N_VISTAS, "budget_vista": BUDGET,
       "shard": SHARD, "brazo": BRAZO, "r": R_LORA, "alpha": ALPHA}
for i, k in enumerate(MIS_KEYS):
    if time.time() - T00 > WALL_MAX:
        print("[s%d] WALL_MAX alcanzado, corto en %d/%d" % (SHARD, i, len(MIS_KEYS)),
              flush=True)
        break
    task = CHAL[k]
    gt = SOLS[k][0]
    pares = [(p["input"], p["output"]) for p in task["train"]]
    seed = int(hashlib.md5(k.encode()).hexdigest()[:8], 16) % 1024**2

    vistas_med = build_vistas(pares, seed + 31, N_VIS_MED + 1)   # [0] = canonica
    celdas = len(gt) * len(gt[0])
    fila = {"task": k, "coste": coste(task), "celdas": celdas, "pool_vacio_en_strat": k in VACIAS,
            "variantes": {}}
    for var in VARIANTES:
        if time.time() - T00 > WALL_MAX:
            print("[s%d] WALL_MAX antes de %s en %s" % (SHARD, var["n"], k), flush=True)
            break
        set_peft_model_state_dict(MODEL, {kk: v.clone() for kk, v in INIT.items()})
        v_train = build_vistas_train(pares, seed + int(var.get("so", 0)), int(var.get("np", 16)))
        t0 = time.time()
        try:
            n_seqs = entrenar_ttt(task, v_train, lr=float(var.get("lr", LR)), epocas=int(var.get("ep", 1)),
                                  mask_all=bool(var.get("mask_all", False)), seed=seed + int(var.get("so", 0)))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); MODEL.zero_grad(set_to_none=True); MODEL.eval()
            fila["variantes"][var["n"]] = {"oom_ttt": True}; continue
        t_ttt = time.time() - t0
        meds = []
        for v in vistas_med:
            try:
                meds.append(rangos_de_gt(task, gt, v))
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); meds.append(None)
            except Exception as e:
                print("[s%d] rangos fallo %s %s: %s" % (SHARD, k, var["n"], str(e)[:100]), flush=True); meds.append(None)
        ok = [m for m in meds if m]
        defs = [m["deficit_vs_greedy"] for m in ok]
        fila["variantes"][var["n"]] = {"n_seqs": n_seqs, "ttt_s": round(t_ttt, 1), "vistas": meds,
                                       "deficit_canon": meds[0]["deficit_vs_greedy"] if meds[0] else None,
                                       "deficit_min": min(defs) if defs else None,
                                       "deficit_med": sorted(defs)[len(defs) // 2] if defs else None,
                                       "n_vistas_greedy": sum(1 for m in ok if m["n_no_argmax"] == 0),
                                       "nll_canon": meds[0]["nll_total"] if meds[0] else None}
        print("[s%d %d/%d] %s %-9s seqs %d ttt %.0fs | deficit canon %s min %s med %s | vistas greedy %d/%d | nll %s"
              % (SHARD, i + 1, len(MIS_KEYS), k, var["n"], n_seqs, t_ttt,
                 fila["variantes"][var["n"]]["deficit_canon"], fila["variantes"][var["n"]]["deficit_min"],
                 fila["variantes"][var["n"]]["deficit_med"], fila["variantes"][var["n"]]["n_vistas_greedy"], len(ok),
                 fila["variantes"][var["n"]]["nll_canon"]), flush=True)
        torch.cuda.empty_cache()
    RES["tareas"].append(fila)
    json.dump(RES, open(OUT, "w"))          # persistencia INCREMENTAL por tarea

json.dump(RES, open(OUT, "w"))
print("[s%d] fin %.0fs" % (SHARD, time.time() - T00), flush=True)


In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
SUCIAS = ["0934a4d8", "136b0064", "16b78196", "981571dc", "aa4ec2a5", "da515329"]
VACIAS = ["21897d95", "53fb4810", "4a21e3da", "221dfab4", "b10624e5", "195c6913"]
MODO = os.environ.get("TTTK_MODO", "completo")
if MODO == "piloto":
    # 4 tareas, 1 por GPU, a lo largo de la distribucion (146-472 s de TTT): casi/casi/cerca/vacia
    PLAN = [["4a21e3da"], ["d59b0160"], ["faa9f03d"], ["28a6681f"]]
    VARIANTES = [{"n": "base"}, {"n": "base_s2", "so": 1}, {"n": "lr1e-4", "lr": 1e-4}]
    WALL_MAX = 1550
else:
    PLAN = [["faa9f03d", "53fb4810", "58f5dbd5", "195c6913"], ["dd6b8c4b", "f931b4a8", "7b3084d4", "89565ca0", "b10624e5"],
            ["3a25b0d8", "b6f77b65", "4a21e3da", "221dfab4"], ["28a6681f", "21897d95", "d59b0160", "67e490f4", "cb2d8a2c"]]
    VARIANTES = json.loads(os.environ.get("TTTK_VARIANTES", '[{"n": "base"}, {"n": "base_s2", "so": 1}, {"n": "lr1e-4", "lr": 1e-4}, {"n": "ep2", "ep": 2}, {"n": "np32", "np": 32}]'))
    WALL_MAX = int(os.environ.get("TTTK_WALL", "11000"))
print("modo", MODO, "variantes", [v["n"] for v in VARIANTES], "WALL_MAX", WALL_MAX, flush=True)

procs = []
for i in range(4):
    env = dict(os.environ)
    env.update({"CUDA_VISIBLE_DEVICES": str(i), "SHARD": str(i), "N_SHARDS": "4", "N_VISTAS": "0",
                "BUDGET_VISTA": "45", "WALL_MAX": str(WALL_MAX), "VARIANTES": json.dumps(VARIANTES),
                "VACIAS": json.dumps(VACIAS), "SUCIAS": json.dumps(SUCIAS), "MIS_KEYS": json.dumps(PLAN[i])})
    fh = open("/kaggle/working/log%d.txt" % i, "w")
    procs.append((i, subprocess.Popen([sys.executable, "/kaggle/working/_worker_src.py"],
                                      env=env, stdout=fh, stderr=subprocess.STDOUT), fh))
    print("worker %d: %s" % (i, PLAN[i]), flush=True)
for i, p, fh in procs:
    p.wait(); fh.close()
    print("worker %d rc=%d a los %.0fs" % (i, p.returncode, time.time() - T0), flush=True)

tareas = []
for i in range(4):
    f = "/kaggle/working/shard%d.json" % i
    if os.path.exists(f):
        tareas += json.load(open(f)).get("tareas", [])
json.dump({"modo": MODO, "variantes": VARIANTES, "tareas": tareas}, open("/kaggle/working/tttk.json", "w"))
for i in range(4):
    print("=" * 12 + (" LOG WORKER %d " % i) + "=" * 12, flush=True)
    try:
        print(open("/kaggle/working/log%d.txt" % i).read()[-3000:], flush=True)
    except Exception as e:
        print("sin log:", e, flush=True)

tareas.sort(key=lambda t: t["coste"])
noms = [v["n"] for v in VARIANTES]
print("")
print("=" * 110, flush=True)
print("TTTK: deficit de la GT frente al greedy (vista canonica / minimo de 4 vistas) por variante del TTT", flush=True)
print("=" * 110, flush=True)
print("%-10s %6s %6s | " % ("tarea", "coste", "pool") + " | ".join("%16s" % n for n in noms), flush=True)
for t in tareas:
    cel = []
    for n in noms:
        r = t["variantes"].get(n)
        if not r or r.get("oom_ttt"): cel.append("%16s" % "-"); continue
        cel.append("%6s/%5s g%d" % (r["deficit_canon"], r["deficit_min"], r["n_vistas_greedy"]))
    print("%-10s %6d %6s | " % (t["task"], t["coste"], "VACIO" if t["pool_vacio_en_strat"] else "pool") + " | ".join(cel), flush=True)
print("")
print("PARES frente a base (negativo = la variante acerca la GT al greedy):", flush=True)
def med(xs):
    s = sorted(xs); k = len(s)
    return s[k // 2] if k % 2 else (s[k // 2 - 1] + s[k // 2]) / 2.0
for n in noms[1:]:
    dc, dm = [], []
    for t in tareas:
        b, r = t["variantes"].get("base"), t["variantes"].get(n)
        if b and r and b.get("deficit_canon") is not None and r.get("deficit_canon") is not None:
            dc.append(r["deficit_canon"] - b["deficit_canon"]); dm.append(r["deficit_min"] - b["deficit_min"])
    if not dc: continue
    print("  %-9s n=%d | canon: mediana %+.2f media %+.2f mejora %d/%d | min4: mediana %+.2f mejora %d/%d"
          % (n, len(dc), med(dc), sum(dc) / len(dc), sum(1 for x in dc if x < 0), len(dc),
             med(dm), sum(1 for x in dm if x < 0), len(dm)), flush=True)
print("")
print("LECTURA: base_s2 = misma config con otra semilla => el ruido del instrumento; una variante", flush=True)
print("         solo cuenta si su delta pareado sale de esa banda en la mayoria de tareas.", flush=True)
print("RESULT_JSON:" + json.dumps({"tareas": len(tareas), "wall_s": round(time.time() - T0)}), flush=True)
